In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

# 1. Cargar los datos limpios que creamos en el paso anterior
df = pd.read_csv('../data/processed/metrics_clean.csv')

# 2. Agrupar los datos por campaña para tener los totales
# Sumamos impresiones, clics y órdenes de todos los días para cada campaña
df_grouped = df.groupby('campaign_id')[['impressions', 'clicks', 'orders']].sum().reset_index()

# Ordenar por impresiones para elegir las dos campañas más grandes para la prueba
df_grouped = df_grouped.sort_values(by='impressions', ascending=False).head(2)

# Mostrar la tabla en pantalla para verificar los datos seleccionados
display(df_grouped)

,campaign_id,impressions,clicks,orders
1,39889,1068337427,420003,1566
7,983498,137806768,509992,313


In [5]:
# Extraer los datos de las dos campañas (Campaña A y Campaña B)
camp_a_id = df_grouped.iloc[0]['campaign_id']
camp_b_id = df_grouped.iloc[1]['campaign_id']

impresiones = [df_grouped.iloc[0]['impressions'], df_grouped.iloc[1]['impressions']]
clics = [df_grouped.iloc[0]['clicks'], df_grouped.iloc[1]['clicks']]
ordenes = [df_grouped.iloc[0]['orders'], df_grouped.iloc[1]['orders']]

print(f"--- ENFRENTAMIENTO: Campaña {camp_a_id} vs Campaña {camp_b_id} ---\n")

--- ENFRENTAMIENTO: Campaña 39889 vs Campaña 983498 ---



In [11]:
# ==========================================
# PRUEBA 1: Z-Test para el CTR (Click-Through Rate)
# ¿Es la diferencia en la tasa de clics estadísticamente significativa?
# ==========================================
stat_z, p_valor_z = proportions_ztest(count=clics, nobs=impresiones)

ctr_a = (clics[0] / impresiones[0]) * 100
ctr_b = (clics[1] / impresiones[1]) * 100

print(f"CTR Campaña A: {ctr_a:.2f}% | CTR Campaña B: {ctr_b:.2f}%")
print(f"P-Valor del Z-Test: {p_valor_z:.5f}")

if p_valor_z < 0.05:
    ganador = "Campaña A" if ctr_a > ctr_b else "Campaña B"
    print(f"✅ Resultado: Con un 95% de confianza, la {ganador} es superior en generar Clics.\n")
else:
    print("⚖️ Resultado: No hay diferencia estadística significativa en el CTR. Fue casualidad.\n")

CTR Campaña A: 0.04% | CTR Campaña B: 0.37%
P-Valor del Z-Test: 0.00000
✅ Resultado: Con un 95% de confianza, la Campaña B es superior en generar Clics.



In [12]:
# ==========================================
# PRUEBA 2: Chi-Cuadrado para Tasa de Conversión (CR)
# ¿Es la diferencia en la tasa de conversión estadísticamente significativa?
# ==========================================
# Creamos una tabla de contingencia: [ [Órdenes A, No-Órdenes A], [Órdenes B, No-Órdenes B] ]
no_ordenes_a = clics[0] - ordenes[0]
no_ordenes_b = clics[1] - ordenes[1]

tabla_contingencia = np.array([
    [ordenes[0], no_ordenes_a],
    [ordenes[1], no_ordenes_b]
])

chi2, p_valor_chi2, dof, expected = chi2_contingency(tabla_contingencia)

cr_a = (ordenes[0] / clics[0]) * 100
cr_b = (ordenes[1] / clics[1]) * 100

print(f"Conversion Rate Campaña A: {cr_a:.2f}% | Conversion Rate Campaña B: {cr_b:.2f}%")
print(f"P-Valor del Chi-Cuadrado: {p_valor_chi2:.4f}")

if p_valor_chi2 < 0.05:
    ganador_cr = "Campaña A" if cr_a > cr_b else "Campaña B"
    print(f"✅ Resultado: Con un 95% de confianza, la {ganador_cr} es superior en generar Ventas.")
else:
    print("⚖️ Resultado: No hay diferencia estadística significativa en la Conversión.")

Conversion Rate Campaña A: 0.37% | Conversion Rate Campaña B: 0.06%
P-Valor del Chi-Cuadrado: 0.0000
✅ Resultado: Con un 95% de confianza, la Campaña A es superior en generar Ventas.
